# Maps of Taunton and Worcester area

## Code setup

In [1]:
import os
import geopandas
import numpy as np
import pandas as pd
import copy
# Plotting:
import matplotlib.pyplot as plt
import cmasher as cmr  # for additional colormaps
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as ticker

import classes.plot_maps as plot_maps

In [2]:
dir_data = 'data'
dir_output = 'output'
dir_images = 'images'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base'
file_vlim_summary = 'vlims_summary_taunworc.csv'
# Geometry:
file_england_lsoa = 'Lower_layer_super_output_areas_(E+W)_2011_Boundaries_(Generalised_Clipped)_V2.zip'
file_uk_outlines = 'Countries_(December_2022)_GB_BFC.zip'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'

## Load data

Colour limits for maps:

In [3]:
p = os.path.join(dir_output, file_vlim_summary)
df_vlim_orig = pd.read_csv(p, index_col=0)

In [4]:
display(df_vlim_orig)

,min,max
column,,
change_in_drip_ship_mt_time,-136.0,0.000
change_in_drip_ship_mt_travel_time,-76.0,0.000
change_in_drip_ship_mt_travel_time_with_delay,-136.0,0.000
change_in_lvo_drip_ship_ivt_mt_mrs_0-2,0.0,0.086
drip_ship_mt_time,170.0,354.000
drip_ship_mt_travel_time,0.0,124.000
drip_ship_mt_travel_time_with_delay,0.0,160.000
drip_ship_mt_travel_time_with_delay_base,19.0,184.000


## Geometry

In [5]:
# Load the England and Wales LSOA shapefile
filename = os.path.join(dir_data, file_england_lsoa)
lsoa_gdf = geopandas.read_file(filename)

# Set or convert to British National Grid (EPSG:27700)
if lsoa_gdf.crs is None:
    lsoa_gdf = lsoa_gdf.set_crs("EPSG:27700")
else:
    lsoa_gdf = lsoa_gdf.to_crs("EPSG:27700")

In [6]:
# Get the England and Wales outline
filename = os.path.join(dir_data, file_uk_outlines)
countries_gdf = geopandas.read_file(filename)

# Set or convert to British National Grid (EPSG:27700)
if countries_gdf.crs is None:
    countries_gdf = countries_gdf.set_crs("EPSG:27700")
else:
    countries_gdf = countries_gdf.to_crs("EPSG:27700")

# Only keep England and Wales outlines:
mask = countries_gdf["CTRY22NM"].isin(["England", "Wales"])
countries_gdf = countries_gdf.loc[mask]
england = countries_gdf.boundary

In [7]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
hospitals_df = pd.read_csv(p)

hospitals_gdf = geopandas.GeoDataFrame(
    hospitals_df,
    geometry=geopandas.points_from_xy(
        pd.to_numeric(hospitals_df["long"]),
        pd.to_numeric(hospitals_df["lat"]),
    ),
    crs="EPSG:4326",
).to_crs("EPSG:27700")

# Only keep hospitals in England:
hospitals_gdf = hospitals_gdf.loc[hospitals_gdf["Country"] == "England"]

display(hospitals_gdf.head(3))

,SSNAP name,Postcode,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT,geometry
0,John Radcliffe Hospital,OX39DU,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1,POINT (453940.001 207531)
1,Royal Berkshire Hospital,RG15AN,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0,POINT (472411.001 172961.999)
2,Wycombe General Hospital,HP112TT,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0,POINT (486386.001 192694)


## Draw map

Make the plots:

In [8]:
def main_plot(data, hospitals_gdf, map_dict, savename='', show=True):
    # Setup for arrows:
    base_combinations = (
        data[['nearest_ivt_unit_base', 'transfer_unit_base']]
        .drop_duplicates()
    )
    comparator_combinations = (
        data[['nearest_ivt_unit_taunworc', 'transfer_unit_taunworc']]
        .drop_duplicates()
    )

    fig = plt.figure(figsize=(8.5, 12))
    gs = GridSpec(3, 5, width_ratios=[40, 1, 10, 40, 1], wspace=0.0, hspace=0.35)
    
    row_ax = 0
    col_ax = 0
    axs = []
    for map_label, ax_dict in map_dict.items():
        if row_ax == 0:
            # Top row axis uses all column space:
            ax = fig.add_subplot(gs[row_ax, :])
        else:
            ax = fig.add_subplot(gs[row_ax, col_ax])
        axs.append(ax)

        if 'lsoa' in ax_dict.keys():
            # Colourbar setup:
            if row_ax == 2 and col_ax == 0:
                ax_dict['lsoa']['cax'] = None
            else:
                ax_dict['lsoa']['cax'] = fig.add_subplot(gs[row_ax, col_ax + 1])
            # Plot main map data.
            plot_maps.plot_map(ax, data, **ax_dict['lsoa'])
        else:
            pass

        if (row_ax == 1) & (col_ax == 0):
            cax = ax_dict['lsoa']['cax']
            cax.yaxis.set_major_locator(ticker.MultipleLocator(30))
        # Hospital markers:
        plot_maps.plot_hospitals(ax, hospitals_gdf,
                                 show_new_mt=ax_dict['markers_show_new_mt'],
                                 show_labels=ax_dict['markers_show_labels'])
        # Draw arrows:
        if ax_dict['arrows'] == 'base':
            for _, row in base_combinations.iterrows():
                plot_maps.draw_arrow(ax, hospitals_gdf, row['nearest_ivt_unit_base'],
                           row['transfer_unit_base'])
        elif ax_dict['arrows'] == 'changed':
            for _, row in comparator_combinations.iterrows():
                destination = plot_maps.get_hospital(hospitals_gdf,
                                           row['transfer_unit_taunworc'])
                if (destination is not None and
                    destination["extra_MT_unit"] == 1):
                    plot_maps.draw_arrow(ax, hospitals_gdf,
                               row['nearest_ivt_unit_taunworc'],
                               row['transfer_unit_taunworc'])
        else:
            pass

        # Axis title:
        ax.set_title(ax_dict['ax_title'], fontsize=12)
                
        col_ax += 3
        if row_ax == 0:
            row_ax +=1
            col_ax =0
        elif col_ax > 5:
            col_ax = 0
            row_ax += 1
    
    # Basemaps:
    for ax in axs:
        plot_maps.add_basemap(ax)

    # England outline:
    for ax in axs:
        england.plot(ax=ax, edgecolor="k", facecolor="#00000000", linewidth=1,
                     zorder=0)

    
    # Figure setup:    
    for ax in axs:
        ax.set_xlim(min_lng, max_lng)
        ax.set_ylim(min_lat, max_lat)
        ax.apply_aspect()
        ax.set_axis_off()

    # Save and/or display picture:
    if len(savename) > 0:
        fig.savefig(p, dpi=300, bbox_inches="tight")
    else:
        pass
    if show:
        plt.show()
    else:
        plt.close()

Setup:

In [9]:
def make_map_dicts(df_vlim):
    map_dict = {
        'units_only': {
            'markers_show_new_mt': False,
            'markers_show_labels': True,
            'arrows': 'none',
            'ax_title': "Stroke units",
        },
        'time_to_mt_unit': {
            'lsoa': dict(
                column='drip_ship_mt_travel_time_with_delay_base',
                cmap="gist_heat_r",
                missing_kwds={"color": "lightgrey"},
                vmin=df_vlim.loc['drip_ship_mt_travel_time_with_delay_base', 'min'],
                vmax=df_vlim.loc['drip_ship_mt_travel_time_with_delay_base', 'max'],
                antialiased=False,
                linewidth=0,
                legend=True,
                legend_kwds=dict(
                    shrink=0.5,
                    pad=0.01,
                    fraction=0.05,
                    label=("Time (minutes)"),
                ),
            ),
            'markers_show_new_mt': False,
            'markers_show_labels': False,
            'arrows': 'base',
            'ax_title': "Time to MT unit\nincluding non-travel delay\n(base scenario)"
        },
        'change_in_time_to_mt_unit': {
            'lsoa': dict(
                column='change_in_drip_ship_mt_time',
                cmap="cmr.pepper",
                missing_kwds={"color": "lightgrey"},
                vmin=df_vlim.loc['change_in_drip_ship_mt_time', 'min'],
                vmax=df_vlim.loc['change_in_drip_ship_mt_time', 'max'],
                antialiased=False,
                linewidth=0,
                legend=True,
                legend_kwds=dict(
                    shrink=0.5,
                    pad=0.01,
                    fraction=0.05,
                    label=("Improvement in time (minutes)"),
                ),
            ),
            'markers_show_new_mt': True,
            'markers_show_labels': False,
            'arrows': 'changed',
            'ax_title': ("Improvement in time to MT unit\n"
                         "including non-travel delay,\n(comparator - base)")
        },
        'outcome_improvement': {
            'lsoa': dict(
                column='change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
                cmap="inferno",
                missing_kwds={"color": "lightgrey"},
                vmin=df_vlim.loc['change_in_lvo_drip_ship_ivt_mt_mrs_0-2', 'min'],
                vmax=df_vlim.loc['change_in_lvo_drip_ship_ivt_mt_mrs_0-2', 'max'],
                antialiased=False,
                linewidth=0,
                legend=False,
            ),
            'markers_show_new_mt': True,
            'markers_show_labels': False,
            'arrows': 'changed',
            'ax_title': ("Improvement in mRS 0-2 outcomes\n"
                         "for treated LVO (comparator - base)")
        },
        'outcome_improvement_lossy': {
            'lsoa': dict(
                column='change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67',
                cmap="inferno",
                missing_kwds={"color": "lightgrey"},
                vmin=df_vlim.loc['change_in_lvo_drip_ship_ivt_mt_mrs_0-2', 'min'],
                vmax=df_vlim.loc['change_in_lvo_drip_ship_ivt_mt_mrs_0-2', 'max'],
                antialiased=False,
                linewidth=0,
                legend=True,
                legend_kwds=dict(
                    shrink=0.5,
                    pad=0.01,
                    fraction=0.05,
                    label=("Improvement in mRS 0-2 outcomes (%)"),
                ),
            ),
            'markers_show_new_mt': True,
            'markers_show_labels': False,
            'arrows': 'changed',
            'ax_title': ("Improvement in mRS 0-2 outcomes\n"
                         "for treated LVO (comparator - base)\n"
                         "(33% reduction in patients transferred)")
        },
    }

    return map_dict

## Plot maps:

In [10]:
for s_bias in ['', '_bias10min']:
    p = os.path.join(dir_output, file_results_comparison + s_bias + '.csv')
    comparison = pd.read_csv(p, index_col='LSOA')

    print('Data columns:')
    print(comparison.columns)
    
    # Limit to LSOA that have changed with the opening
    # of a Worcester or Taunton unit:
    comparison = comparison[comparison['changed']]
    
    # Check hospitals used here:
    used_hospitals = sorted(list(set(comparison[[
        'transfer_unit_base', 'transfer_unit_taunworc',
        'nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc'
    ]].dropna().values.flatten())))
    # Get all postcodes from comparisonMT units
    comparitor_mt_postcodes = (
        comparison['transfer_unit_taunworc'].unique().tolist())
    
    print('Used hospital postcodes:', used_hospitals)
    print('New MT units:', comparitor_mt_postcodes)

    # Limit hospitals_gdf to only those used in the comparison
    hospitals_gdf = hospitals_gdf.loc[hospitals_gdf['Postcode'].isin(used_hospitals)]
    # Find comparitor_mt_postcodes not labelled as Use_MT = 1
    # in the hospitals_gdf
    extra_mt_postcodes = list(
        set(hospitals_gdf[hospitals_gdf['Use_MT'] == 0]['Postcode'].tolist()) &
        set(comparitor_mt_postcodes)
    )
    # Create an extra column in the hospitals_gdf to indicate if a
    # hospital is an extra MT unit
    hospitals_gdf['extra_MT_unit'] = (
        hospitals_gdf['Postcode'].isin(extra_mt_postcodes).astype(int))

    
    # Merge the results with the LSOA shapefile
    comparison = pd.merge(lsoa_gdf, comparison, left_on='LSOA11NM',
                          right_on='area_lower', how='right')
    # Make sensible axis limits:
    min_lng, min_lat, max_lng, max_lat = plot_maps.get_bounds(
        comparison, hospitals_gdf, used_hospitals)
    print('Axis limits:', (min_lng, min_lat, max_lng, max_lat))
    # Custom axis limits so that bias and non-bias maps share extent:
    min_lng, min_lat, max_lng, max_lat = (
        202711.5960517817, 49563.999368048535,
        449793.0008166602, 303793.90588181384
    )
    print('Manual axis limits:', (min_lng, min_lat, max_lng, max_lat))
    
    # Manually remove Hospital_City for Dudley if in used_hospital list
    hosp_reject = ['DY12HQ', 'CV22DX']
    if any([u in used_hospitals for u in hosp_reject]):
        # Remove row
        hospitals_gdf = hospitals_gdf[
            ~hospitals_gdf['Postcode'].isin(hosp_reject)]
    
    # Adjust data for plotting:
    data = comparison.copy(deep=True)
    data['change_in_lvo_drip_ship_ivt_mt_mrs_0-2'] *= 100
    data['change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67'] *= 100
    data['change_in_drip_ship_mt_time'] *= -1
    data['change_in_drip_ship_mt_travel_time'] *= -1
    
    # Adjust colour limits to match:
    df_vlim = df_vlim_orig.copy()
    for c in ['change_in_drip_ship_mt_travel_time', 'change_in_drip_ship_mt_time']:
        v1 = df_vlim.loc[c, 'min']
        v2 = df_vlim.loc[c, 'max']
        df_vlim.loc[c, 'max'] = -v1
        df_vlim.loc[c, 'min'] = -v2
    
    df_vlim.loc['change_in_lvo_drip_ship_ivt_mt_mrs_0-2'] = (
        df_vlim.loc['change_in_lvo_drip_ship_ivt_mt_mrs_0-2'] * 100)
    
    # Check colour limits of the data here:
    cols_plotted = [
        'drip_ship_mt_travel_time_with_delay_base',
        'change_in_drip_ship_mt_time',
        'change_in_drip_ship_mt_travel_time',
        'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
        'change_in_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67'
    ]
    print('Data limits (min, max):')
    for col in cols_plotted:
        vmin_col = data[col].min(axis=None)
        vmax_col = data[col].max(axis=None)
        print(col, vmin_col, vmax_col)
    
    map_dict = make_map_dicts(df_vlim)

    output_name = 'taunworc'
    file_maps = f'comparison_map_four_in_one{s_bias}_{output_name}.png'
    p = os.path.join(dir_images, file_maps)
    
    main_plot(data, hospitals_gdf, map_dict, savename=p, show=False)

Data columns:
Index(['area_lower', 'Admissions_2025_Est', 'nearest_ivt_unit_base',
       'transfer_unit_base', 'transfer_required_base',
       'drip_ship_mt_time_base', 'drip_ship_mt_travel_time_base',
       'drip_ship_mt_travel_time_with_delay_base', 'admissions_mt_base',
       'transfers_mt_base', 'admissions_mt_transferloss0!67_base',
       'transfers_mt_transferloss0!67_base',
       'admissions_weighted_drip_ship_mrs_0-2_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67_base',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base',
       'drip_ship_mrs_0-2_base', 'lvo_drip_ship_ivt_mt_mrs_0-2_base',
       'drip_ship_mrs_0-2_transferloss0!67_base',
       'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67_base',
       'nearest_ivt_unit_taunworc', 'transfer_unit_taunworc',
       'transfer_required_taunworc', 'drip_ship_mt_time_taunworc',
       'drip_ship_mt_travel_time_